# Text Classification with SVMs

In this notebook, we use SVMs to find who wrote a text: a TF-IDF representation and a linear SVM, as on the "SVM and Text" slide.

The corpus comes from classical English literature: 9 authors, 2 books per author, a file per chapter.

1. Read the corpus and represent it as TF-IDF.
2. Evaluate a linear SVM, then `LinearSVC`, tuning `C`.
3. Understand why these scores are misleading, and evaluate by book.
4. Test two other hypotheses: the length of the texts and the proper nouns.
5. Read the model's errors and the words that make it decide.

Let's start by getting the corpus, and the spaCy model used later on:

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm

In [ ]:
import pathlib
import time
from collections.abc import Iterable
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import sklearn.feature_extraction.text
import sklearn.metrics
import sklearn.model_selection
import sklearn.pipeline
import sklearn.svm
import spacy
import tqdm.notebook

## Building the Training Set

After a quick look at the corpus files (`classical-authors/<author>/<book>/<chapter>.txt`), write the `get_data` function, which takes the corpus path and returns three lists of the same length, one item per chapter:

- the texts;
- their author;
- their book (used later on).

You should get 1,214 chapters, 9 authors and 18 books.

**Useful documentation**

- [`pathlib.Path.glob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob)
- [`pathlib.Path.read_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)
- [`pathlib.PurePath.parent`](https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.parent)

In [ ]:
def get_data(directory: pathlib.Path) -> tuple[list[str], list[str], list[str]]:
  texts = []
  authors = []
  books = []

  # Your code here

  return texts, authors, books


texts, authors, books = get_data(pathlib.Path("classical-authors"))
print(len(texts), "chapters,", len(set(authors)), "authors,", len(set(books)), "books")

### Additional hints

The calls to use:

- `directory.glob("*/*/*.txt")`: every chapter, three levels below the corpus (`sorted` puts them in the same order on every machine);
- for a chapter `path`, `path.parent.name` is its book and `path.parent.parent.name` its author;
- `path.read_text(encoding="utf8")` reads the whole file.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def get_data(directory: pathlib.Path) -> tuple[list[str], list[str], list[str]]:
  texts = []
  authors = []
  books = []
  for path in sorted(directory.glob(...)):
    texts.append(...)
    authors.append(...)
    books.append(...)
  return texts, authors, books
```

### Solution

In [ ]:
def get_data(directory: pathlib.Path) -> tuple[list[str], list[str], list[str]]:
  texts = []
  authors = []
  books = []
  for path in sorted(directory.glob("*/*/*.txt")):
    texts.append(path.read_text(encoding="utf8"))
    authors.append(path.parent.parent.name)
    books.append(path.parent.name)
    # Variant: path.parts[-3] and path.parts[-2]
  return texts, authors, books


texts, authors, books = get_data(pathlib.Path("classical-authors"))
print(len(texts), "chapters,", len(set(authors)), "authors,", len(set(books)), "books")

## TF-IDF Representation

Turn `texts` into a TF-IDF matrix `X` with scikit-learn's `TfidfVectorizer`: a row per chapter, a column per word of the vocabulary. Display its shape and the share of its nonzero values. You should get a shape of `(1214, 39623)` and 1.4% nonzero values.

**Useful documentation**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Your code here

### Additional hints

`vectorizer.fit_transform(texts)` learns the vocabulary and returns a sparse matrix: `X.shape` gives its shape, `X.nnz` its number of nonzero values.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(...)
print(X.shape)
print(X.nnz / ...)
```

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(texts)
print(X.shape)
print(f"{X.nnz / (X.shape[0] * X.shape[1]):.1%} nonzero values")

## Training and Evaluation

Evaluate a `SVC` with a linear kernel by 5-fold cross-validation (`cross_val_score`), on `X` and `authors`: scikit-learn's classifiers accept text labels, no need to encode them. Display the scores and the computation time.

**Useful documentation**

- [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
- [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
# Your code here

### Additional hints

- `sklearn.svm.SVC(kernel="linear")`: the linear kernel SVM;
- `cross_val_score(model, X, authors, cv=5)` returns a score (the accuracy) per fold;
- `time.perf_counter()`, called before and after, measures the duration in seconds.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
start = time.perf_counter()
scores = sklearn.model_selection.cross_val_score(..., X, authors, cv=...)
print(time.perf_counter() - start)
print(scores, scores.mean())
```

### Solution

In [ ]:
start = time.perf_counter()
scores = sklearn.model_selection.cross_val_score(
  sklearn.svm.SVC(kernel="linear"), X, authors, cv=5
)
print(f"Cross-validation : {time.perf_counter() - start:.1f} s")
print(scores, scores.mean())

## `LinearSVC` and Tuning `C`

`LinearSVC` isn't a faster implementation of the same model, but a very close model, optimized for the linear kernel (squared hinge loss, one vs all: see the `svm-basics` demo, "Multi-Class SVM"). Its main setting is `C`, the soft margin.

- The vectorizer, fitted above on all the texts, has seen the test folds (their words count in the IDF). Put it with the classifier in a pipeline (`make_pipeline`): it is refitted on the training folds only, at each fold.
- Tune `C` with a 5-fold grid search (`GridSearchCV`) over the values `np.logspace(-2, 2, 5)`.

Display the score of each `C`, the best one, and the duration of the search: compare it with the `SVC`'s, for 5 times as many trainings.

**Useful documentation**

- [`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)
- [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [ ]:
# Your code here

### Additional hints

- `make_pipeline(TfidfVectorizer(), LinearSVC())` names each step after its class, in lowercase: the grid parameter is written `"linearsvc__C"`;
- the pipeline takes the raw texts: `grid.fit(texts, authors)`;
- `grid.cv_results_["mean_test_score"]` gives the mean score of each `C`, `grid.best_params_` and `grid.best_score_` the best one;
- `n_jobs=-1` spreads the trainings over all the cores.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
pipeline = sklearn.pipeline.make_pipeline(
  sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
)
grid = sklearn.model_selection.GridSearchCV(
  pipeline, {"linearsvc__C": ...}, cv=5, n_jobs=-1
)
grid.fit(..., ...)
print(grid.cv_results_["mean_test_score"])
print(grid.best_params_, grid.best_score_)
```

### Solution

In [ ]:
pipeline = sklearn.pipeline.make_pipeline(
  sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
)
grid = sklearn.model_selection.GridSearchCV(
  pipeline, {"linearsvc__C": np.logspace(-2, 2, 5)}, cv=5, n_jobs=-1
)
start = time.perf_counter()
grid.fit(texts, authors)
print(f"Grid search : {time.perf_counter() - start:.1f} s")
print(grid.cv_results_["mean_test_score"])
print(grid.best_params_, grid.best_score_)

## Conclusions

What can we conclude from the scores?

### Solution

Scores this close to 1 are suspicious. Three hypotheses:

- chapters of the same book are both in the training and the test folds: the model may recognize the book (its characters, places, subject) rather than the author;
- the texts are long (whole chapters): a few words specific to the author are enough;
- the names of characters and places, mentioned in almost every chapter, give the book away.

To test them: evaluate by training on one book of each author and testing on the other; evaluate at the sentence level; replace the named entities.

## Evaluating by Book

Train on one book of each author and test on the other: that is the real situation, attributing a text the model has never seen, not another chapter of a book it knows.

- Write a function `get_book_index(authors, books)` that gives each chapter the number of its book for its author, 0 or 1.
- Evaluate the pipeline with `cross_val_score`, `GroupKFold(n_splits=2)` and these numbers as groups: one fold trains on books 0 and tests on books 1, the other does the reverse.

What becomes of the score? What did the 5-fold cross-validation measure?

**Useful documentation**

- [`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)

In [ ]:
# Your code here

### Additional hints

- Why not `groups=books`? `GroupKFold(n_splits=2)` would split the 18 books into two packs of similar sizes, regardless of the authors: both books of an author may land in the same fold, and that author is then never seen in training (a score around 0.41).
- The first book of an author, in alphabetical order: go through `sorted(set(zip(authors, books)))` and keep the first book seen for each author (`dict.setdefault`).
- `cross_val_score(pipeline, texts, authors, cv=..., groups=...)`: the pipeline of the previous part, with `C` at its default value.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def get_book_index(authors: list[str], books: list[str]) -> list[int]:
  first_book = {}
  for author, book in sorted(set(zip(authors, books))):
    first_book.setdefault(author, book)
  return [... for author, book in zip(authors, books)]


book_index = get_book_index(authors, books)
scores = sklearn.model_selection.cross_val_score(
  pipeline,
  texts,
  authors,
  cv=sklearn.model_selection.GroupKFold(n_splits=...),
  groups=...,
)
print(scores, scores.mean())
```

### Solution

In [ ]:
def get_book_index(authors: list[str], books: list[str]) -> list[int]:
  first_book = {}
  for author, book in sorted(set(zip(authors, books))):
    first_book.setdefault(author, book)
  return [int(book != first_book[author]) for author, book in zip(authors, books)]


book_index = get_book_index(authors, books)
scores = sklearn.model_selection.cross_val_score(
  pipeline,
  texts,
  authors,
  cv=sklearn.model_selection.GroupKFold(n_splits=2),
  groups=book_index,
)
print(f"By book : {scores} {scores.mean():.3f}")

The score falls from 0.99 to 0.80: the 5-fold cross-validation mixed the chapters of a book between training and test, and mostly measured the recognition of a book already seen. 0.80 is the score to expect on an unknown book, still far above chance (1 in 9). The `C` chosen above was chosen on these same mixed folds: it too is to be tuned by book.

## Tuning the Representation

Vary the settings of `TfidfVectorizer` in the pipeline and compare the scores by book:

- `sublinear_tf=True`: $1 + \log(\text{tf})$ instead of the raw frequency;
- `min_df=2`: ignores the words of a single chapter;
- `ngram_range=(1, 2)`: adds the pairs of consecutive words (slower).

Which one helps? Why, do you think?

In [ ]:
# Your code here

### Additional hints

A loop over dictionaries of parameters, passed through `TfidfVectorizer(**params)`; `{}` keeps the default settings, to compare.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for params in [{}, {"sublinear_tf": True}, ...]:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(**params), sklearn.svm.LinearSVC()
  )
  scores = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=book_index,
  )
  print(params, scores.mean())
```

### Solution

In [ ]:
for params in [
  {},
  {"sublinear_tf": True},
  {"min_df": 2},
  {"ngram_range": (1, 2)},
]:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(**params), sklearn.svm.LinearSVC()
  )
  scores = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=book_index,
    n_jobs=-1,
  )
  print(params, f"{scores.mean():.3f}")

By book, `sublinear_tf=True` clearly wins (0.80 → 0.86): with the logarithm, a word repeated dozens of times in a chapter (a character's name, the subject of the book) no longer outweighs the words the author uses in all their texts. `min_df=2` barely changes anything (0.81); `ngram_range=(1, 2)` does a little worse (0.78), for a much slower training: the pairs of words mostly add phrases specific to a book.

## Replacing Named Entities With Special Markers

To replace the names of characters and places (and more generally all the named entities), we use `spacy`, with its English model downloaded at the start. We don't need most of the components of its pipeline: we disable them, and enable the only additional component we will need (the sentence splitter).

In [ ]:
nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")

To test the proper nouns hypothesis, let's start with a function that replaces the named entities with their type. Here are the entities spaCy finds in a sentence:

In [ ]:
doc = nlp("Jimi Hendrix was absolutely amazing at Woodstock!")

# Display the named entities
print(
  " ".join(f"{t}/{t.ent_type_}/{t.ent_iob_}" if t.ent_type_ else t.text for t in doc)
)

### The `replace_ners` Function

If the output above is:

    Jimi/PERSON/B Hendrix/PERSON/I was absolutely amazing at Woodstock/ORG/B !

the `replace_ners` function must return:

    PPPERSON was absolutely amazing at OOORG !

We triple the first letter of the type: it's a trick to avoid producing a word already in the vocabulary. Use the `ent_type_` and `ent_iob_` attributes of the tokens.

**Useful documentation**

- [`spacy.tokens.Token`, its attributes](https://spacy.io/api/token#attributes)

In [ ]:
def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> str:
  # Your code here
  return ""


replace_ners(doc)

#### Additional hints

- `token.ent_iob_` is `"B"` on the first token of an entity, `"I"` on the next ones and `"O"` outside entities: keep the text of the `"O"`s, a marker for each `"B"`, and nothing for the `"I"`s;
- `token.ent_type_` gives the type of the entity (`"PERSON"`, `"ORG"`…);
- `token.text` is the text of the token.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> str:
  result = []
  for token in tokens:
    if token.ent_iob_ == ...:
      result.append(token.text)
    elif token.ent_iob_ == ...:
      result.append(...)
  return " ".join(result)
```

#### Solution

In [ ]:
def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> str:
  result = []
  for token in tokens:
    if token.ent_iob_ == "O":
      result.append(token.text)
    elif token.ent_iob_ == "B":
      result.append(f"{token.ent_type_[0] * 2}{token.ent_type_}")
  return " ".join(result)


print(replace_ners(doc))

## Extended `get_data` Function

Here is the `get_data` function reworked to return the texts at two scales, the sentence and the chapter (document), each with and without the named entities replaced, with the author and the book of each text. It returns a `Data` object with 8 fields: 4 versions of the texts, the authors and the books at both scales. It runs the whole corpus through spaCy: allow a few minutes.

In [ ]:
class Data(NamedTuple):
  texts_sentence_replaced_ners: list[str]
  texts_sentence_normal: list[str]
  texts_document_replaced_ners: list[str]
  texts_document_normal: list[str]
  authors_sentence: list[str]
  authors_document: list[str]
  books_sentence: list[str]
  books_document: list[str]


def get_data(directory: pathlib.Path) -> Data:
  authors_sentence = []
  authors_document = []
  books_sentence = []
  books_document = []
  texts_sentence_replaced_ners = []
  texts_sentence_normal = []
  texts_document_replaced_ners = []
  texts_document_normal = []
  paths = sorted(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(
    zip(paths, nlp.pipe(contents, n_process=1)), total=len(paths)
  ):
    author = path.parent.parent.name
    book = path.parent.name
    for sentence in doc.sents:
      authors_sentence.append(author)
      books_sentence.append(book)
      texts_sentence_replaced_ners.append(replace_ners(sentence))
      texts_sentence_normal.append(" ".join(token.text for token in sentence))
    authors_document.append(author)
    books_document.append(book)
    texts_document_replaced_ners.append(replace_ners(doc))
    texts_document_normal.append(" ".join(token.text for token in doc))
  return Data(
    texts_sentence_replaced_ners=texts_sentence_replaced_ners,
    texts_sentence_normal=texts_sentence_normal,
    texts_document_replaced_ners=texts_document_replaced_ners,
    texts_document_normal=texts_document_normal,
    authors_sentence=authors_sentence,
    authors_document=authors_document,
    books_sentence=books_sentence,
    books_document=books_document,
  )


data = get_data(pathlib.Path("classical-authors"))

## Training and Evaluating the Configurations

What are now the scores of the different configurations (document or sentence, with or without the named entities replaced), with 5-fold cross-validation and by book?

Write a function `evaluate(texts, authors, books)` that returns these two mean scores for a TF-IDF + `LinearSVC` pipeline, and call it on the 4 configurations. At the sentence level, allow a few minutes.

What do you conclude about the three hypotheses?

In [ ]:
# Your code here

### Additional hints

The two evaluations of the previous parts, in a function: `get_book_index(authors, books)` gives the groups of the evaluation by book.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def evaluate(
  texts: list[str], authors: list[str], books: list[str]
) -> tuple[float, float]:
  pipeline = ...
  mixed = sklearn.model_selection.cross_val_score(pipeline, texts, authors, cv=5)
  by_book = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=get_book_index(..., ...),
  )
  return mixed.mean(), by_book.mean()


print(evaluate(data.texts_document_normal, data.authors_document, data.books_document))
```

### Solution

In [ ]:
def evaluate(
  texts: list[str], authors: list[str], books: list[str]
) -> tuple[float, float]:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
  )
  mixed = sklearn.model_selection.cross_val_score(
    pipeline, texts, authors, cv=5, n_jobs=-1
  )
  by_book = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=get_book_index(authors, books),
    n_jobs=-1,
  )
  return mixed.mean(), by_book.mean()


for scale, texts_normal, texts_replaced, authors_, books_ in [
  (
    "document",
    data.texts_document_normal,
    data.texts_document_replaced_ners,
    data.authors_document,
    data.books_document,
  ),
  (
    "sentence",
    data.texts_sentence_normal,
    data.texts_sentence_replaced_ners,
    data.authors_sentence,
    data.books_sentence,
  ),
]:
  for name, texts_ in [
    ("without replacing the entities", texts_normal),
    ("with the entities replaced", texts_replaced),
  ]:
    mixed, by_book = evaluate(texts_, authors_, books_)
    print(f"{scale:<9} {name:<31} 5 folds {mixed:.3f}, by book {by_book:.3f}")

At the document level: 0.99 with 5 folds and 0.81 by book without replacing the entities, 0.99 and 0.85 with. At the sentence level: 0.64 and 0.39 without replacing them, 0.58 and 0.40 with.

- **The book** is the main cause: by book, the score loses almost 0.2. The mixed folds mostly measured the recognition of a book already seen.
- **The length** matters too: a sentence carries far fewer clues than a chapter (0.39 by book), but stays 3 to 4 times above chance (0.11): the style shows in a few words already.
- **The proper nouns** didn't explain the 0.99: replacing them barely changes anything with mixed folds at the document level. By book, replacing them *helps* (0.81 → 0.85): the characters of a book don't appear in the other one, they are noise for the model. With mixed folds at the sentence level, removing them costs 0.06: a name is enough to recognize the book of a sentence.

## Confusion Matrix

On the chapters with the entities replaced, train a pipeline on books 0 and predict the author of the chapters of books 1 (the numbers of `get_book_index`). Display the accuracy, then the confusion matrix normalized by row, with the display names of `DISPLAY_NAMES`.

What does the diagonal show? Which authors does the model confuse?

**Useful documentation**

- [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

In [ ]:
# Display names (the corpus folders keep their typos)
DISPLAY_NAMES = {
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "wilde": "Wilde",
}

# Your code here

### Additional hints

- As NumPy arrays (`np.array(..., dtype=object)` for the texts), `texts_[book_index_ == 0]` selects the chapters of books 0;
- `normalize="true"` divides each row (a true author) by its total: the diagonal gives the share of their chapters correctly attributed;
- `labels=pipeline.classes_` sets the order of the authors, and `display_labels` their displayed names, in the same order.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
texts_ = np.array(data.texts_document_replaced_ners, dtype=object)
authors_ = np.array(data.authors_document)
book_index_ = np.array(get_book_index(data.authors_document, data.books_document))

pipeline = ...
pipeline.fit(texts_[book_index_ == 0], ...)
predicted = pipeline.predict(...)

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  ...,
  predicted,
  labels=pipeline.classes_,
  display_labels=[DISPLAY_NAMES[author] for author in pipeline.classes_],
  normalize="true",
)
plt.show()
```

### Solution

In [ ]:
texts_ = np.array(data.texts_document_replaced_ners, dtype=object)
authors_ = np.array(data.authors_document)
book_index_ = np.array(get_book_index(data.authors_document, data.books_document))

pipeline = sklearn.pipeline.make_pipeline(
  sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
)
pipeline.fit(texts_[book_index_ == 0], authors_[book_index_ == 0])
predicted = pipeline.predict(texts_[book_index_ == 1])
print("Accuracy on books 1:", (predicted == authors_[book_index_ == 1]).mean())

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  authors_[book_index_ == 1],
  predicted,
  labels=pipeline.classes_,
  display_labels=[DISPLAY_NAMES[author] for author in pipeline.classes_],
  normalize="true",
  values_format=".2f",
  xticks_rotation="vertical",
  cmap="Blues",
)
plt.show()

Accuracy: about 0.85. Each row is a true author, and the diagonal the share of their chapters correctly attributed (their recall). The errors make sense: Wilde is taken for Shaw in almost a third of the cases (two playwrights: the dialogue form weighs more than the author), Charlotte Brontë for her sister Anne (12%), Thoreau and Emerson, two essayists of the same circle, are a little confused.

## Analyzing the Resulting Model

`LinearSVC` does one vs all: our model is made of 9 binary classification SVMs, one per author against all the others, in the order of `classes_`.

Its `coef_` attribute holds their 9 weight vectors, of the size of the vocabulary: an author's score for a document is the dot product of their vector with the document as TF-IDF (plus a constant). The words of largest weight are the ones that tip the most toward that author.

With `numpy.argsort`, we display the most discriminating words of each author against all the others.

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
n_best = 10
vectorizer, model = pipeline[0], pipeline[-1]
for i, author in enumerate(model.classes_):
  author_vec = model.coef_[i]
  index = np.argsort(-author_vec)[:n_best]
  top_words = vectorizer.get_feature_names_out()[index]
  bestwords = (
    f"{word} ({score:.2f})" for score, word in zip(author_vec[index], top_words)
  )
  print(f"{DISPLAY_NAMES[author]:<17} {', '.join(bestwords)}")

Which words stand out? What do they tell about what the model recognizes?

### Solution

Three kinds of words:

- function words (*the*, *his*, *of* for Cooper; *or*, *is* for Emerson; *but*, *and*, *my*, *upon* for Anne Brontë; *our*, *we* for Thoreau): the author's style, which carries over from one book to the other;
- words of the subject of the book (*whale*, *ship*, *sea* for Melville; *scout*, *rifle*, *fort* for Cooper; *river*, *stream* for Thoreau);
- proper nouns spaCy's small model didn't recognize (*heyward*, *queequeg*, *pickering*, *higgins*, *illingworth*…), and the markers themselves (*ppperson*, *ooorg*, *dddate*): the frequency of named entities is a clue too.

The subject words and the names don't carry over from one book to the other: they are what inflated the mixed evaluation. Stylometry, which attributes texts to their authors, actually relies mostly on the frequency of function words.